# Article18 reproduction notebook

This notebook is the reviewer-facing entry point. It verifies the article claims against the kept CSV reports and the Python implementation, then runs a small protected visual search demo. Full metric regeneration commands are provided as optional cells because they require pretrained vision backbones and can take substantially longer than the claim verifier.

In [1]:
from pathlib import Path
import subprocess
import sys

try:
    import torch  # noqa: F401
    import torchvision  # noqa: F401
except ModuleNotFoundError as exc:
    raise RuntimeError("Run this notebook in the Python environment where `pip install -r requirements.txt` was executed.") from exc

ROOT = Path.cwd()
print(f"Working directory: {ROOT}")

Working directory: /home/d.yacenko/work/Articles/Article18_Perceptual_Search_Crypto_Verification_REVIEWER_PACKAGE


## 1. Check the archived metric reports

The verifier checks the numeric claims used in the paper against `reports/*.csv` and also tests the secret-geometry helper implementation.

In [2]:
reports = sorted((ROOT / "reports").glob("*.csv"))
print(f"CSV reports: {len(reports)}")
for path in reports:
    print(" -", path.relative_to(ROOT))

CSV reports: 18
 - reports/protected_gardens_point_day_right_night_right_densenet121_regionalg4_bits4096_b64_w41_o0_mb16_mt1000_sg1_leakage_summary.csv
 - reports/protected_gardens_point_day_right_night_right_densenet121_regionalg4_bits4096_b64_w41_o0_mb16_mt1000_sg1_pair_summary.csv
 - reports/protected_gardens_point_day_right_night_right_densenet121_regionalg4_bits4096_b64_w41_o0_mb16_mt1000_sg1_threshold_sweep.csv
 - reports/protected_gardens_point_day_right_night_right_dinov2_vits14_cls_bits4096_b64_w41_o0_mb16_mt1000_sg1_leakage_summary.csv
 - reports/protected_gardens_point_day_right_night_right_dinov2_vits14_cls_bits4096_b64_w41_o0_mb16_mt1000_sg1_pair_summary.csv
 - reports/protected_gardens_point_day_right_night_right_dinov2_vits14_cls_bits4096_b64_w41_o0_mb16_mt1000_sg1_threshold_sweep.csv
 - reports/protected_gardens_point_day_right_night_right_dinov2_vits14_regionalg4_bits4096_b64_w41_o0_mb16_mt1000_sg0_leakage_summary.csv
 - reports/protected_gardens_point_day_right_night_

In [3]:
cmd = [sys.executable, "-B", "scripts/verify_article_claims.py"]
result = subprocess.run(cmd, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f"Claim verification failed with exit code {result.returncode}")

OK: verified 92 article claims against CSV artifacts and Python implementation.



## 2. Run a small end-to-end protected search demo

The demo creates a small protected database from Gardens Point reference images, discards open descriptors and binary visual codes from the stored records, and scans the protected records with query images.

In [4]:
RUN_DEMO = True

if RUN_DEMO:
    cmd = [
        sys.executable,
        "-B",
        "scripts/demo_protected_visual_search.py",
        "--database-size", "12",
        "--query-count", "3",
        "--batch-size", "4",
        "--min-components", "6",
    ]
    result = subprocess.run(cmd, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Demo failed with exit code {result.returncode}")

Using cache found in /home/d.yacenko/.cache/torch/hub/facebookresearch_dinov2_main
/home/d.yacenko/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/d.yacenko/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/d.yacenko/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Protected visual matching demo
  dataset: gardens_point
  database sequence: day_right, records: 12
  query sequence: night_right, queries: 3
  acceptance rule: at least 6 verified components
  helper geometry: secret

Encoding images with DINOv2...
Binarizing transient visual components...
Creating protected DB records...

Prote

## 3. Optional full experiment regeneration

Set `RUN_FULL_EXPERIMENTS = True` to regenerate the main DINOv2 protected experiment and the backbone comparison reports. This can require CUDA and downloading pretrained model weights if they are not already cached.

In [ ]:
RUN_FULL_EXPERIMENTS = False

full_commands = [
    [sys.executable, "-B", "scripts/run_place_keyed_component_experiment.py"],
    [sys.executable, "-B", "scripts/run_place_keyed_component_experiment.py", "--secret-geometry"],
    [sys.executable, "-B", "scripts/run_place_keyed_component_experiment.py", "--component-mode", "cls", "--secret-geometry"],
    [sys.executable, "-B", "scripts/run_place_torchvision_component_experiment.py", "--model-name", "vit_b_16", "--secret-geometry"],
    [sys.executable, "-B", "scripts/run_place_torchvision_component_experiment.py", "--model-name", "resnet50", "--secret-geometry"],
    [sys.executable, "-B", "scripts/run_place_torchvision_component_experiment.py", "--model-name", "densenet121", "--secret-geometry"],
]

for cmd in full_commands:
    print(" ".join(cmd))
    if RUN_FULL_EXPERIMENTS:
        result = subprocess.run(cmd, cwd=ROOT, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"Experiment failed with exit code {result.returncode}: {' '.join(cmd)}")

## 4. Optional LaTeX build

The package includes the compiled PDF and the LaTeX sources. Set `RUN_LATEX_BUILD = True` to rebuild the PDF locally with XeLaTeX.

In [ ]:
RUN_LATEX_BUILD = False

if RUN_LATEX_BUILD:
    latex_dir = ROOT / "latex"
    tex_file = "article18_protected_visual_matching_ru.tex"
    for _ in range(2):
        result = subprocess.run(
            ["xelatex", "-interaction=nonstopmode", tex_file],
            cwd=latex_dir,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        print(result.stdout[-4000:])
        if result.returncode != 0:
            raise RuntimeError(f"LaTeX build failed with exit code {result.returncode}")
    print(latex_dir / "article18_protected_visual_matching_ru.pdf")